# 02 · Run the persona experiment

Every story is run under five conditions, and only the persona changes: no persona, mismatched nationality, matched nationality, full persona (nationality plus sex, age and field of study), and a length-matched persona that pads the matched one with a neutral sentence.

In [16]:
import json, os, random, time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from types import SimpleNamespace
import pandas as pd

## Settings

In [ ]:
# MODEL = "gpt-4o-mini"
# BASE_URL = None
# API_KEY = os.environ.get("OPENAI_API_KEY")
MODEL = "llama3.2:3b"
BASE_URL = "http://localhost:11434/v1"
API_KEY = "ollama"
BACKEND = "api"            # api = Ollama API; local = local llama.cpp
STORIES = "data/stories.csv"
OUT = Path("results/raw.jsonl")
LIMIT = None #20                 # pilot first; None = all stories
WORKERS = 2 #8
SEED = 2026
args = SimpleNamespace(model=MODEL, backend=BACKEND, seed=SEED)

## Prompts and personas

In [ ]:
EMOTIONS = ["joy", "fear", "anger", "sadness", "disgust", "shame", "guilt"]
TASK = ("Here is a situation you experienced:\n\"{text}\"\n\n"
        "What is the main emotion you felt in this situation? Answer with exactly one word "
        "from this list: joy, fear, anger, sadness, disgust, shame, guilt.")
NEUTRAL = ["Answer honestly.", "Please answer honestly.", "Please read this carefully.",
           "Please read the situation carefully.", "Please read the situation below carefully.",
           "Please read the situation below carefully before answering.",
           "Please read the situation below carefully and then answer honestly."]


def nationality_persona(country):
    return f"You are a person from {country}."


def full_persona(row):
    parts = [f"You are a {int(row.age)}-year-old" if pd.notna(row.age) else "You are a",
             row.sex if isinstance(row.sex, str) else "person",
             f"from {row.country}"]
    text = " ".join(parts)
    if isinstance(row.field, str):
        text += f" who studies {row.field}"
    return text + "."


def length_matched_persona(row):
    """Matched nationality persona padded with neutral sentences to the full persona's length."""
    base = nationality_persona(row.country)
    gap = len(full_persona(row).split()) - len(base.split())
    if gap <= 0:
        return base
    filler = min(NEUTRAL, key=lambda s: abs(len(s.split()) - gap))
    return base + " " + filler


def build_jobs(stories):
    countries = sorted(stories.country.unique())
    jobs = []
    for row in stories.itertuples(index=False):
        others = [c for c in countries if c != row.country]
        personas = { "none": "You are a person.",
                    "matched": nationality_persona(row.country),
                    "full": full_persona(row),
                    "length_matched": length_matched_persona(row)}
        for other in others:  # with 2 countries: exactly one mismatched persona
            personas[f"mismatched::{other}"] = nationality_persona(other)
        for cond, persona in personas.items():
            condition, _, persona_country = cond.partition("::")
            if condition in ("matched", "full", "length_matched"):
                persona_country = row.country
            jobs.append({"story_id": row.story_id, "writer_id": row.writer_id,
                         "source_country": row.country, "gold": row.emotion,
                         "condition": condition, "persona_country": persona_country or "",
                         "persona": persona or "", "prompt": TASK.format(text=row.text),
                         "key": f"{row.story_id}|{cond}"})
    return jobs


## Look at the five prompts for one story


In [19]:
stories = pd.read_csv(STORIES)
if LIMIT:
    stories = stories.head(LIMIT)
jobs = build_jobs(stories)
first = stories.story_id.iloc[0]
for j in jobs:
    if j["story_id"] == first:
        print(f'{j["condition"]:<15} ({len(j["persona"].split()):>2} words) {j["persona"] or "(none)"}')
print("\n" + jobs[0]["prompt"])

none            ( 4 words) You are a person.
matched         ( 6 words) You are a person from Spain.
full            (10 words) You are a 20-year-old woman from Spain who studies psychology.
length_matched  (10 words) You are a person from Spain. Please read this carefully.
mismatched      ( 6 words) You are a person from Zambia.

Here is a situation you experienced:
"I was drinking coffee in a cafe and I found a piece of scrubber in my cup."

What is the main emotion you felt in this situation? Answer with exactly one word from this list: joy, fear, anger, sadness, disgust, shame, guilt.


## Calling the model

In [ ]:
def call_model(job, args, client):
    if args.backend == "mock": 
        rng = random.Random(job["key"])
        return job["gold"] if rng.random() < 0.6 else rng.choice(EMOTIONS)
    messages = ([{"role": "system", "content": job["persona"]}] if job["persona"] else [])
    messages.append({"role": "user", "content": job["prompt"]})
    for attempt in range(6):
        try:
            r = client.chat.completions.create(model=args.model, messages=messages,
                                               temperature=0, max_tokens=10, seed=args.seed)
            return r.choices[0].message.content or ""
        except Exception as e:  # rate limits, timeouts
            wait = 2 ** attempt
            print(f"  retry {attempt + 1} in {wait}s: {e}")
            time.sleep(wait)
    return "__ERROR__"


In [ ]:
lines = [l for l in OUT.read_text().splitlines() if l.strip() and '"condition": "none"' not in l]
OUT.write_text("\n".join(lines) + "\n")
print(len(lines), "rows kept (should be 4224)")

4224 rows kept (should be 4224)


## Run

In [22]:
OUT.parent.mkdir(parents=True, exist_ok=True)
done = set()
if OUT.exists():
    done = {json.loads(l)["key"] for l in OUT.read_text().splitlines() if l.strip()}
todo = [j for j in jobs if j["key"] not in done]
print(f"{len(jobs)} calls in total, {len(done)} already done, {len(todo)} to run.")

client = None
if BACKEND == "api":
    from openai import OpenAI
    client = OpenAI(api_key=API_KEY, base_url=BASE_URL)

with OUT.open("a") as f, ThreadPoolExecutor(WORKERS) as pool:
    futures = {pool.submit(call_model, j, args, client): j for j in todo}
    for i, fut in enumerate(as_completed(futures), 1):
        j = {**futures[fut], "model": MODEL if BACKEND == "api" else "mock", "raw_output": fut.result()}
        j.pop("prompt")
        f.write(json.dumps(j) + "\n"); f.flush()
        if i % 250 == 0:
            print(f"  {i}/{len(todo)}")
print("Done:", OUT)

5280 calls in total, 4224 already done, 1056 to run.
  250/1056
  500/1056
  750/1056
  1000/1056
Done: results\raw.jsonl


## Pilot check

Every output should be a single label from the list. If you see sentences, refusals or other words, fix the prompt before the full run and note the change in the pre-registration.

In [23]:
raw = pd.read_json(OUT, lines=True)
print(raw.raw_output.str.strip().str.lower().value_counts().head(15))
raw.groupby("condition").size()

raw_output
sadness.                                                  1088
shame.                                                     772
fear.                                                      706
joy.                                                       614
guilt.                                                     609
anger.                                                     545
disgust.                                                   404
anger                                                       61
i would say the main emotion i felt in this                 46
i cannot provide an answer that may be associated with      24
disgust                                                     18
joy                                                         15
i would choose the word "shame".                            14
i cannot provide an answer that justifies or promotes       12
i would choose: fear.                                       11
Name: count, dtype: int64


condition
full              1056
length_matched    1056
matched           1056
mismatched        1056
none              1056
dtype: int64